# 090. Wydajność, Big Data i Techniki Out-of-Core

Triki optymalizacyjne w Pandas: optymalizacja RAM typem `category`, downcasting typów numerycznych, przetwarzanie strumieniowe w partiach (`chunksize`) oraz analityczny silnik SQL **DuckDB** bezpośrednio na plikach.


## 1. Optymalizacja pamięci RAM: konwersja na typ `category`


In [ ]:
import pandas as pd
import numpy as np


In [ ]:

# Tworzymy dużą tabelę z powtarzalnymi napisami (500 000 wierszy)
n = 500_000
miasta = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław', 'Poznań']
df_duza = pd.DataFrame({
    'Miasto': np.random.choice(miasta, size=n),
    'Wartosc': np.random.uniform(10, 1000, size=n)
})

pamiec_przed = df_duza['Miasto'].memory_usage(deep=True) / (1024**2)
print(f"Pamięć kolumny Miasto (zwykły object): {pamiec_przed:.2f} MB")


In [ ]:

# Konwersja na category:
df_duza['Miasto'] = df_duza['Miasto'].astype('category')
pamiec_po = df_duza['Miasto'].memory_usage(deep=True) / (1024**2)
print(f"Pamięć kolumny Miasto po konwersji na category: {pamiec_po:.2f} MB")
print(f"Zysk pamięciowy: {(1 - pamiec_po/pamiec_przed)*100:.1f}% oszczędności RAM!")


## 2. Strumieniowanie w partiach (`chunksize`)


In [ ]:
# Symulacja: zapisujemy plik testowy
df_duza.head(50_000).to_csv('duzy_plik.csv', index=False)

# Strumieniowanie partiami po 10 000 wierszy bez przeciążania pamięci:
laczna_suma = 0.0
licznik_wierszy = 0

for chunk in pd.read_csv('duzy_plik.csv', chunksize=10_000):
    laczna_suma += chunk['Wartosc'].sum()
    licznik_wierszy += len(chunk)

print(f"Przetworzono {licznik_wierszy} wierszy strumieniowo.")
print(f"Łączna wyliczona suma: {laczna_suma:,.2f}")


## 3. Szybki analityczny SQL na plikach w DuckDB


In [ ]:
try:
    import duckdb
    # Zapytanie analityczne bezpośrednio na pliku CSV bez wczytywania przez Pandas!
    wynik_sql = duckdb.query("""
        SELECT Miasto, COUNT(*) as Liczba, ROUND(AVG(Wartosc), 2) as Srednia
        FROM 'duzy_plik.csv'
        GROUP BY Miasto
        ORDER BY Srednia DESC
    """).df()
    print("Wynik zapytania DuckDB bezpośrednio na dysku:")
    print(wynik_sql)
except ImportError:
    print("DuckDB nie jest zainstalowane w środowisku - zainstaluj: pip install duckdb")
